Plotting global OHC anomalies and global-mean SST anomalies over time for 45˚ latitude-bounded 70-yr ramp experiments.

In [ ]:
%run /home/Kiera.Lowman/Kd-sensitivity-analysis/notebooks/read_and_calculate.ipynb # to get selecting_basins function
%run /home/Kiera.Lowman/Kd-sensitivity-analysis/notebooks/plotting_functions.ipynb
%run /home/Kiera.Lowman/Kd-sensitivity-analysis/notebooks/compute_ensemble_means.ipynb
%run /home/Kiera.Lowman/Kd-sensitivity-analysis/notebooks/time_series_plotting.ipynb

In [ ]:
profiles = ['surf','therm','mid','bot']
prof_strings = ["Surf","Therm","Mid","Bot"]

power_inputs = ['0.1TW', '0.2TW', '0.3TW']
power_var_suff = ['0p1TW', '0p2TW', '0p3TW']
power_strings = ['0.1 TW', '0.2 TW', '0.3 TW']

p10_power_inputs = ['0.019TW','0.036TW']
p10_power_var_suff = ['0p019TW','0p036TW']
p10_power_strings = ['0.019 TW','0.036 TW']

scalar_pp_type = 'ts-annual'
scalar_vars = ["thetaoga","sst_global"]
scalar_diag_file = 'ocean_scalar_monthly'

ramp_scalar_ts_dir = "/home/Kiera.Lowman/Python_figures/45deg_70yr_time_series/"

# Reading data

## Data for the select SURF and THERM experiments

In [ ]:
time_chunk = 25

start = 1
end = 200

for idx, prof in enumerate(profiles[0:2]):
    get_ens_diff_and_means('const',time_chunk,start,end,
                           scalar_vars,
                           pp_type=scalar_pp_type,diag_file=scalar_diag_file,
                           profiles=[prof],
                           power_inputs=[p10_power_inputs[idx]],power_var_suff=[p10_power_var_suff[idx]],
                           lat_bound=45,ramp_exp=True,
                           num_ens_mem=6,omit_mem1=True,
                           verbose=True)
    
    ds_name = f"const_{prof}_{p10_power_var_suff[idx]}_{start}_{end}"
    myVars[ds_name]["OHC"] = calc_OHC_anom(myVars[ds_name].thetaoga)
    diff_ds_name = f"{ds_name}_diff"
    myVars[diff_ds_name]["OHC"] = calc_OHC_anom(myVars[diff_ds_name].thetaoga)

    arr_name = f"const_{prof}_{p10_power_var_suff[idx]}_{start}_{end}_diff_mem_list"
    for ens_idx in range(5):
        myVars[arr_name][ens_idx]["OHC"] = calc_OHC_anom(myVars[arr_name][ens_idx]["thetaoga"])

## Data for standard power inputs

In [ ]:
time_chunk = 25

start = 1
end = 200

get_ens_diff_and_means('const+doub',time_chunk,start,end,
                       scalar_vars,
                       pp_type=scalar_pp_type,diag_file=scalar_diag_file,
                       lat_bound=45,ramp_exp=True,
                       num_ens_mem=6,omit_mem1=True,
                       verbose=True)

### Calculate OHC for ensemble members

In [ ]:
# members: calculate OHC
for co2_scen in ['const','doub']:
    arr_name = f"{co2_scen}_ctrl_{start}_{end}_mem_list"
    for ens_idx in range(5):
        myVars[arr_name][ens_idx]["OHC"] = calc_OHC_anom(myVars[arr_name][ens_idx]["thetaoga"])
        
for co2_scen in ['const','doub']:
    for prof in profiles:
        for power in power_var_suff:
            arr_name = f"{co2_scen}_{prof}_{power}_{start}_{end}_mem_list"
            for ens_idx in range(5):
                myVars[arr_name][ens_idx]["OHC"] = calc_OHC_anom(myVars[arr_name][ens_idx]["thetaoga"])

In [ ]:
# members: calculate OHC differences
arr_name = f"doub_ctrl_{start}_{end}_diff_mem_list"
for ens_idx in range(5):
    myVars[arr_name][ens_idx]["OHC"] = calc_OHC_anom(myVars[arr_name][ens_idx]["thetaoga"])

for diff_type in ['const','1860','2xctrl','const_ctrl']:
    for prof in profiles:
        for power in power_var_suff:
            if diff_type == 'const':
                arr_name = f"const_{prof}_{power}_{start}_{end}_diff_mem_list"
            else:
                arr_name = f"doub_{prof}_{power}_{start}_{end}_diff_{diff_type}_mem_list"
            for ens_idx in range(5):
                myVars[arr_name][ens_idx]["OHC"] = calc_OHC_anom(myVars[arr_name][ens_idx]["thetaoga"])

### Calculate OHC for ensemble mean

In [ ]:
# mean: calculate OHC
for co2_scen in ['const','doub']:
    ds_name = f"{co2_scen}_ctrl_{start}_{end}"
    myVars[ds_name]["OHC"] = calc_OHC_anom(myVars[ds_name].thetaoga)
    for prof in profiles:
        for power in power_var_suff:
            ds_name = f"{co2_scen}_{prof}_{power}_{start}_{end}"
            myVars[ds_name]["OHC"] = calc_OHC_anom(myVars[ds_name].thetaoga)

In [ ]:
# mean: calculate OHC differences
for co2_scen in ['const','doub']:
    if co2_scen == 'doub':
        ds_name = f"{co2_scen}_ctrl_{start}_{end}_diff"
        myVars[ds_name]["OHC"] = calc_OHC_anom(myVars[ds_name].thetaoga)
    for prof in profiles:
        for power in power_var_suff:
            if co2_scen == 'doub':
                for diff_suff in ['const_ctrl','1860','2xctrl']:
                    ds_name = f"{co2_scen}_{prof}_{power}_{start}_{end}_diff_{diff_suff}"
                    myVars[ds_name]["OHC"] = calc_OHC_anom(myVars[ds_name].thetaoga)
            else:
                ds_name = f"{co2_scen}_{prof}_{power}_{start}_{end}_diff"
                myVars[ds_name]["OHC"] = calc_OHC_anom(myVars[ds_name].thetaoga)

# OHC plots

In [ ]:
start = 1
end = 200

comp_OHC_ystep_list = [200,500,500]
comp_OHC_frac_list = [0.25,0.2,0.2]
comp_OHC_ylims_list = [None,
                       None,
                       [-100,2600]]

tot_OHC_ystep_list = [500,800,800]
tot_OHC_frac_list = [0.2,0.25,0.25]
tot_OHC_ylims_list = [None,
                      None,#[-200,4300],
                      None]

In [ ]:
plot_ts_mixing_diff('doub',ramp_scalar_ts_dir,start,end,fig_pref="45deg_70yr",anom_var='OHC',
                    roll_mean = False, ystep_list = comp_OHC_ystep_list, ymin_frac_list = comp_OHC_frac_list, ylims_list = comp_OHC_ylims_list,
                     leg_loc = 'upper left',
                     leg_ncols = 1,
                   plot_ens_env=True)

In [ ]:
plot_ts_lin_pow_diff('const',ramp_scalar_ts_dir,start,end,fig_pref="45deg_70yr",anom_var='OHC',
                     roll_mean = False, ystep=500,
                    plot_ens_env=True)

In [ ]:
plot_ts_rad_pow_diff('doub',ramp_scalar_ts_dir,start,end,fig_pref="45deg_70yr",anom_var='OHC',
                    roll_mean = False, ystep=500,
                     leg_loc = 'upper left',
                     leg_ncols = 1,
                    plot_ens_env=True)

In [ ]:
plot_ts_mix_pow_diff('const',ramp_scalar_ts_dir,start,end,fig_pref="45deg_70yr",anom_var='OHC',
                     ystep=500, #ylimits=(-100,2600),
                     # profiles=['mid','bot'],omit_ctrl=True,
                     #   power_var_suff = ['0p2TW', '0p3TW'], 
                     #   power_strings = ['0.2 TW', '0.3 TW'],
                     roll_mean = False,
                    plot_ens_env=True)

In [ ]:
plot_lin_ts_diff('doub',ramp_scalar_ts_dir,start,end,fig_pref="45deg_70yr",anom_var='OHC',
                 ystep_list = tot_OHC_ystep_list, ymin_frac_list = tot_OHC_frac_list, ylims_list = tot_OHC_ylims_list,
                     roll_mean = False,
                 plot_ens_env=True
                )

## Percent differences in mixing OHC between CO2 scenarios

In [ ]:
print("const and doub 0.1 TW OHC anomalies at yr 200")
for prof in profiles:
    const_ds = f"const_{prof}_0p1TW_1_200_diff"
    doub_ds = f"doub_{prof}_0p1TW_1_200_diff_2xctrl"
    const_val = myVars[const_ds].OHC.isel(time=-1).values
    doub_val = myVars[doub_ds].OHC.isel(time=-1).values
    per_diff = abs(100*(const_val-doub_val)/((const_val+doub_val)/2))
    raw_diff = doub_val - const_val
    print(f"{prof}: {const_val:.4g}\t{doub_val:.4g}\t per_diff: {per_diff:.4g}")
    # print(f"raw difference: {raw_diff:.4g}\n")

print("const and doub 0.2 TW OHC anomalies at yr 200")
for prof in profiles:
    const_ds = f"const_{prof}_0p2TW_1_200_diff"
    doub_ds = f"doub_{prof}_0p2TW_1_200_diff_2xctrl"
    const_val = myVars[const_ds].OHC.isel(time=-1).values
    doub_val = myVars[doub_ds].OHC.isel(time=-1).values
    per_diff = abs(100*(const_val-doub_val)/((const_val+doub_val)/2))
    raw_diff = doub_val - const_val
    print(f"{prof}: {const_val:.4g}\t{doub_val:.4g}\t per_diff: {per_diff:.4g}")
    # print(f"raw difference: {raw_diff:.4g}\n")

print("const and doub 0.3 TW OHC anomalies at yr 200")
for prof in profiles:
    const_ds = f"const_{prof}_0p3TW_1_200_diff"
    doub_ds = f"doub_{prof}_0p3TW_1_200_diff_2xctrl"
    const_val = myVars[const_ds].OHC.isel(time=-1).values
    doub_val = myVars[doub_ds].OHC.isel(time=-1).values
    per_diff = abs(100*(const_val-doub_val)/((const_val+doub_val)/2))
    raw_diff = doub_val - const_val
    print(f"{prof}: {const_val:.4g}\t{doub_val:.4g}\t per_diff: {per_diff:.4g}")
    # print(f"raw difference: {raw_diff:.4g}\n")

In [ ]:
date_yr70 = cftime.DatetimeNoLeap(70, 7, 2, 12, 0, 0, 0, has_year_zero=True)

print("const and doub 0.1 TW OHC anomalies at yr 70")
for prof in profiles:
    const_ds = f"const_{prof}_0p1TW_1_200_diff"
    doub_ds = f"doub_{prof}_0p1TW_1_200_diff_2xctrl"
    const_val = myVars[const_ds].OHC.sel(time=date_yr70).values
    doub_val = myVars[doub_ds].OHC.sel(time=date_yr70).values
    per_diff = abs(100*(const_val-doub_val)/((const_val+doub_val)/2))
    raw_diff = doub_val - const_val
    print(f"{prof}: {const_val:.4g}\t{doub_val:.4g}\t per_diff: {per_diff:.4g}")
    # print(f"raw difference: {raw_diff:.4g}\n")

print("const and doub 0.2 TW OHC anomalies at yr 70")
for prof in profiles:
    const_ds = f"const_{prof}_0p2TW_1_200_diff"
    doub_ds = f"doub_{prof}_0p2TW_1_200_diff_2xctrl"
    const_val = myVars[const_ds].OHC.sel(time=date_yr70).values
    doub_val = myVars[doub_ds].OHC.sel(time=date_yr70).values
    per_diff = abs(100*(const_val-doub_val)/((const_val+doub_val)/2))
    raw_diff = doub_val - const_val
    print(f"{prof}: {const_val:.4g}\t{doub_val:.4g}\t per_diff: {per_diff:.4g}")
    # print(f"raw difference: {raw_diff:.4g}\n")

print("const and doub 0.3 TW OHC anomalies at yr 70")
for prof in profiles:
    const_ds = f"const_{prof}_0p3TW_1_200_diff"
    doub_ds = f"doub_{prof}_0p3TW_1_200_diff_2xctrl"
    const_val = myVars[const_ds].OHC.sel(time=date_yr70).values
    doub_val = myVars[doub_ds].OHC.sel(time=date_yr70).values
    per_diff = abs(100*(const_val-doub_val)/((const_val+doub_val)/2))
    raw_diff = doub_val - const_val
    print(f"{prof}: {const_val:.4g}\t{doub_val:.4g}\t per_diff: {per_diff:.4g}")
    # print(f"raw difference: {raw_diff:.4g}\n")

## Comparing radiative OHC at 200 yr

In [ ]:
print("Radiative OHC anomalies at yr 200")
print(f"1pct2xCO2 control: {doub_ctrl_1_200_diff.OHC.isel(time=-1).values:5g} ZJ")

doub_vals = []

for prof in profiles:
    doub_val_strings = []
    for power in power_var_suff:
        doub_ds = f"doub_{prof}_{power}_1_200_diff_1860"
        doub_vals.append(myVars[doub_ds].OHC.isel(time=-1).values)
        doub_val_strings.append(f"{doub_vals[-1]:.5g}")
    print(doub_val_strings)
    
print(np.min(doub_vals))
print(np.max(doub_vals))

## Required energy for mixing OHC = 10% radiative OHC

### yr 70

In [ ]:
date_yr70 = cftime.DatetimeNoLeap(70, 7, 2, 12, 0, 0, 0, has_year_zero=True)

for idx, power in enumerate(power_var_suff):
    print(f"Constant CO2 ramp {power_strings[idx]} 70-yr response")
    for prof in profiles:
        ds_name = f"const_{prof}_{power}_1_200_diff"
        OHC_val = myVars[ds_name].OHC.sel(time=date_yr70).values
        print(f"{prof}: {OHC_val:.3g} ZJ")
    print("")

print("\n1pct2xCO2 control 70-yr response")
dt_co2 = doub_ctrl_1_200_diff.OHC.sel(time=date_yr70).values
print(f"{dt_co2:.3g} ZJ")

print("\nRequired energy input for mixing response equal to 100% or 10% of CO2 response")

power_floats = np.array([0.1, 0.2, 0.3])
for idx, power in enumerate(power_var_suff):
    print(f"\nEstimates from {power_strings[idx]} ramp experiments:")
    for prof in profiles:
        mix_resp_name = f"const_{prof}_{power}_1_200_diff"
        dt_mix = myVars[mix_resp_name].OHC.sel(time=date_yr70).values
        
        roc = dt_mix/power_floats[idx]
        req_energy = dt_co2/roc
        p10_req_energy = dt_co2*0.1/roc
        
        print(f"Rate of change: {roc:4g} ZJ per TW; \t{prof}:\t {req_energy:.3g} TW; {p10_req_energy:.3g} TW")

In [ ]:
date_yr70 = cftime.DatetimeNoLeap(70, 7, 2, 12, 0, 0, 0, has_year_zero=True)

for idx, power in enumerate(power_var_suff):
    print(f"1pct2xCO2 ramp {power_strings[idx]} 70-yr response")
    for prof in profiles:
        ds_name = f"doub_{prof}_{power}_1_200_diff_2xctrl"
        OHC_val = myVars[ds_name].OHC.sel(time=date_yr70).values
        print(f"{prof}: {OHC_val:.3g} ZJ")
    print("")

print("\n1pct2xCO2 control 70-yr response")
dt_co2 = doub_ctrl_1_200_diff.OHC.sel(time=date_yr70).values
print(f"{dt_co2:.3g} ZJ")

print("\nRequired energy input for mixing response equal to 100% or 10% of CO2 response")

power_floats = np.array([0.1, 0.2, 0.3])
for idx, power in enumerate(power_var_suff):
    print(f"\nEstimates from {power_strings[idx]} ramp experiments:")
    for prof in profiles:
        mix_resp_name = f"doub_{prof}_{power}_1_200_diff_2xctrl"
        dt_mix = myVars[mix_resp_name].OHC.sel(time=date_yr70).values
        
        roc = dt_mix/power_floats[idx]
        req_energy = dt_co2/roc
        p10_req_energy = dt_co2*0.1/roc
        
        print(f"Rate of change: {roc:.3g} ZJ per TW; \t{prof}:\t {req_energy:.3g} TW; {p10_req_energy:.3g} TW")

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

# 70-yr requirements
values = [0.019, 0.036, 0.9] # , 0
errors = [0.005, 0.005, 0.3] # , 0
colors = ['b', 'm', 'g'] # , 'r'

fig, ax = plt.subplots(figsize=(6, 3))

# --- Main plot ---
bars = ax.barh(
    prof_strings[0:3],
    values,
    xerr=errors,
    capsize=5,
    color=colors
)

ax.set_xlim(0, 2)
ax.set_xlabel("Required Energy Input (TW)")

# Main annotations
padding = 0.04 * (max(values) + max(errors))
for bar, err in zip(bars, errors):
    width = bar.get_width()
    y = bar.get_y() + bar.get_height() / 2
    ax.text(
        width + err + padding, y,
        f"{width:g}",
        va="center",
        ha="left",
        fontweight="bold"
    )

# --- Inset: zoom on the first two tiny bars ---
small_idx = [0, 1]

# Create inset axes inside the main axes
axins = inset_axes(
    ax,
    width="60%",     # size of inset
    height="60%",
    loc="lower right",
    borderpad=1.0
)

# Plot ONLY the small bars in the inset
prof_small = [prof_strings[i] for i in small_idx]
vals_small = [values[i] for i in small_idx]
errs_small = [errors[i] for i in small_idx]
cols_small = [colors[i] for i in small_idx]

bars_inset = axins.barh(
    prof_small,
    vals_small,
    xerr=errs_small,
    capsize=4,
    color=cols_small
)

# Zoomed x-limits: a bit beyond (value + error) for the small bars
xmax_small = max(v + e for v, e in zip(vals_small, errs_small))
axins.set_xlim(0, xmax_small * 1.35)

# Make inset labels readable / compact
axins.tick_params(axis="both", labelsize=8)
axins.set_xticklabels([])
axins.set_xlabel("")  # usually cleaner to omit inset xlabel

# Inset annotations (smaller font)
inset_padding = 0.04 * axins.get_xlim()[1]
for bar, err in zip(bars_inset, errs_small):
    width = bar.get_width()
    y = bar.get_y() + bar.get_height() / 2
    axins.text(
        width + err + inset_padding, y,
        f"{width:.3g}",
        va="center",
        ha="left",
        fontweight="bold",
        fontsize=10
    )

# Draw connectors showing what region is zoomed
mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.5", lw=1)

plt.savefig(
    "/home/Kiera.Lowman/Python_figures/45deg_70yr_bar_charts/45deg_70yr_OHC_yr70_req_bar_chart.pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

## Plot for select power inputs

In [ ]:
plot_ts_diff_select('const',ramp_scalar_ts_dir,start,end,p10_power_var_suff,p10_power_strings,profiles = profiles[0:2],anom_var='OHC',
                    fig_pref="45deg_70yr_10percent",
                    vline_yr=70,
                    roll_mean = False,
                    leg_loc = 'upper left',
                    plot_ens_env=True
                   )

# SST_global plots

In [ ]:
start = 1
end = 200

comp_sst_ystep_list = [0.1,0.2,0.2]
comp_sst_frac_list = [0.25,0.25,0.25]
comp_sst_ylims_list = [[-0.4,0.4],
                       [-0.6,0.6],
                       [-0.8,0.6]]

tot_sst_ystep_list = [0.2,0.2,0.2]
tot_sst_frac_list = [0.25,0.25,0.25]
tot_sst_ylims_list = [[-0.2,1.4],
                      [-0.2,1.6],
                      [-0.2,1.8]]

In [ ]:
plot_ts_mixing_diff('doub',ramp_scalar_ts_dir,start,end,fig_pref="45deg_70yr",anom_var="sst_global",
                    ystep_list = comp_sst_ystep_list, ymin_frac_list = comp_sst_frac_list, ylims_list = comp_sst_ylims_list,
                     leg_loc = 'upper left', leg_ncols = 2,
                   plot_ens_env=True,
                    ens_env_co2_scens=['const'],
                    env_line=True
                   )

In [ ]:
plot_ts_lin_pow_diff('const',ramp_scalar_ts_dir,start,end,anom_var="sst_global",
                     omit_ctrl=True,
                     fig_pref="45deg_70yr",
                     ylimits = [-0.8,0.4], ystep=0.2,
                     leg_loc = 'lower left',
                     # ylimits = [-1.0,1.5]
                    )

In [ ]:
plot_ts_rad_pow_diff('doub',ramp_scalar_ts_dir,start,end,fig_pref="45deg_70yr",anom_var="sst_global",
                    ylimits = [-0.2,1.6], ystep=0.2,
                     leg_loc = 'lower right', leg_ncols = 2,
                     ctrl_lw=4,
                     plot_ens_env=True,
                     plot_ctrl_env=False,
                     ens_env_power=['0p2TW'],
                     env_line=True
                    )

In [ ]:
plot_ts_mix_pow_diff('const',ramp_scalar_ts_dir,start,end,fig_pref="45deg_70yr",anom_var="sst_global",
                     omit_ctrl=True,
                     ylimits = [-0.6,0.6], ystep=0.2,
                     plot_ens_env=True,
                     ens_env_power=['0p2TW'],
                     env_line=True
                    )

In [ ]:
plot_lin_ts_diff('doub',ramp_scalar_ts_dir,start,end,fig_pref="45deg_70yr",anom_var="sst_global",
                 ystep_list = tot_sst_ystep_list, ymin_frac_list = tot_sst_frac_list, ylims_list = tot_sst_ylims_list,
                 leg_loc = 'lower right',
                 ctrl_lw=4,
                 plot_ens_env=True,
                 plot_sum_env=False,
                 plot_ctrl_env=False,
                 env_line=True
                )

## Comparing radiative SST anomaly at yr 200 and yr 70

In [ ]:
roll_mean_window = 10

In [ ]:
print("Radiative SST anomalies at yr 196")
print(f"1pct2xCO2 control: {doub_ctrl_1_200_diff.sst_global.rolling(time=roll_mean_window, center=True).mean().isel(time=-5).values:5g}")
print(f"1pct2xCO2 control: {doub_ctrl_1_200_diff.sst_global.rolling(time=roll_mean_window, center=True).mean().time.values[-5]}")

doub_vals = []

for prof in profiles:
    doub_val_strings = []
    for power in power_var_suff:
        doub_ds = f"doub_{prof}_{power}_1_200_diff_1860"
        doub_vals.append(myVars[doub_ds].sst_global.rolling(time=roll_mean_window, center=True).mean().isel(time=-5).values)
        doub_val_strings.append(f"{doub_vals[-1]:.5g}")
    print(doub_val_strings)
    
print(np.min(doub_vals))
print(np.max(doub_vals))

In [ ]:
date_yr70 = cftime.DatetimeNoLeap(70, 7, 2, 12, 0, 0, 0, has_year_zero=True)

print("Radiative SST anomalies at yr 70")
print(f"1pct2xCO2 control: {doub_ctrl_1_200_diff.sst_global.rolling(time=roll_mean_window, center=True).mean().sel(time=date_yr70).values:5g}")

doub_vals = []

for prof in profiles:
    doub_val_strings = []
    for power in power_var_suff:
        doub_ds = f"doub_{prof}_{power}_1_200_diff_1860"
        doub_vals.append(myVars[doub_ds].sst_global.rolling(time=roll_mean_window, center=True).mean().sel(time=date_yr70).values)
        doub_val_strings.append(f"{doub_vals[-1]:.5g}")
    print(doub_val_strings)
    
print(np.min(doub_vals))
print(np.max(doub_vals))